[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/typer-certified/notebooks/day-03-capstone-production-cli-app.ipynb#scrollTo=a3b4c5d6)

---
# Day 3 · Capstone — Production CLI Application
**certified-journeys / typer-certified** &nbsp;|&nbsp; Capstone

> **Goal for today:** Build `datakit`, a pip-installable CSV inspection CLI with three commands, a full test suite, and a `pyproject.toml` — in 30 minutes.


In [ ]:
%pip install -q 'typer[all]'


---
## Step 1 · Build `datakit` — three commands

We'll build `datakit`, a CLI for inspecting CSV files:

| Command | Does |
|---|---|
| `datakit info FILE` | Row count, column count, column names |
| `datakit head FILE` | First N rows as a Rich table |
| `datakit stats FILE` | Min / max / mean for numeric columns |

Uses in-memory CSV so the notebook is self-contained. Production code is identical — swap `StringIO` for `open(filepath)`.


In [ ]:
import typer, csv, json
from io import StringIO
from typing import Optional
from rich.console import Console
from rich.table import Table
from typer.testing import CliRunner

console = Console()
runner  = CliRunner()
APP_VERSION = '1.0.0'

SAMPLE_CSV = '''name,age,city,salary
Alice,30,Toronto,95000
Bob,25,Vancouver,72000
Carol,35,Toronto,110000
Dave,28,Montreal,68000
Eve,42,Toronto,130000
'''

def _read(filepath: str) -> list[dict]:
    src = StringIO(SAMPLE_CSV) if 'sample' in filepath.lower() else open(filepath)
    return list(csv.DictReader(src))

# -- app --
app = typer.Typer(name='datakit', help='Inspect CSV files.', rich_markup_mode='rich')

def _version_cb(value: bool):
    if value: typer.echo(f'datakit {APP_VERSION}'); raise typer.Exit()

@app.callback()
def main(version: bool = typer.Option(False,'--version','-V',
         callback=_version_cb, is_eager=True, help='Show version')):
    """datakit — CSV file inspector."""

@app.command()
def info(filepath: str = typer.Argument(..., help='CSV file path')):
    """Show row count, column count and names."""
    rows = _read(filepath)
    cols = list(rows[0].keys()) if rows else []
    t = Table(show_header=False, box=None)
    t.add_row("[dim]Rows[/dim]",    str(len(rows)))
    t.add_row("[dim]Columns[/dim]", str(len(cols)))
    t.add_row("[dim]Names[/dim]",   ", ".join(cols))
    console.print(t)

@app.command()
def head(filepath: str = typer.Argument(...),
         n: int = typer.Option(5,'--rows','-n', min=1, max=100)):
    """Show first N rows."""
    rows = _read(filepath)
    if not rows: typer.echo('Empty.'); return
    t = Table(*rows[0].keys())
    for row in rows[:n]: t.add_row(*row.values())
    console.print(t)

@app.command()
def stats(filepath: str = typer.Argument(...),
          fmt: str = typer.Option('table', click_type=__import__('click').Choice(['table','json']))):
    """Show min/max/mean for numeric columns."""
    rows = _read(filepath); out = {}
    for col in rows[0].keys():
        try:
            vals = [float(r[col]) for r in rows if r[col]]
            out[col] = {'min':min(vals),'max':max(vals),'mean':round(sum(vals)/len(vals),2)}
        except ValueError: pass
    if fmt == 'json': typer.echo(json.dumps(out, indent=2)); return
    t = Table('Column','Min','Max','Mean')
    for col, s in out.items(): t.add_row(col, str(s['min']), str(s['max']), str(s['mean']))
    console.print(t)

# smoke test
print(runner.invoke(app, ['info', 'sample.csv']).output)


**What just happened?**
- **`@app.callback()`** is the right place for global options like `--version` — runs before every command
- `_read()` is a private helper shared by all commands — not a CLI command itself
- `console.print(t)` renders a Rich table; `typer.echo(json.dumps(...))` gives a machine-readable alternative
- All three commands use the same `_read()` + Rich table pattern — consistent UX with minimal code


---
## Step 2 · Test suite + pyproject.toml

A production CLI needs tests for every command and a `pyproject.toml` to make it pip-installable:

```toml
[project.scripts]
datakit = "datakit.main:app"   # creates the 'datakit' shell command
```

After `pip install .`, running `datakit` in the shell calls `app` in `datakit/main.py`.


In [ ]:
# -- full test suite --
def test_version():
    r = runner.invoke(app, ['--version'])
    assert r.exit_code == 0 and APP_VERSION in r.output

def test_info():
    r = runner.invoke(app, ['info', 'sample.csv'])
    assert r.exit_code == 0
    assert '5' in r.output and '4' in r.output   # 5 rows, 4 cols

def test_head_default():
    r = runner.invoke(app, ['head', 'sample.csv'])
    assert r.exit_code == 0 and 'Alice' in r.output

def test_head_n2():
    r = runner.invoke(app, ['head', 'sample.csv', '--rows', '2'])
    assert r.exit_code == 0
    assert 'Alice' in r.output
    assert 'Carol' not in r.output   # only 2 rows

def test_stats_json():
    r = runner.invoke(app, ['stats', 'sample.csv', '--fmt', 'json'])
    assert r.exit_code == 0
    data = json.loads(r.output)
    assert 'age' in data and 'salary' in data

def test_stats_invalid_fmt():
    r = runner.invoke(app, ['stats', 'sample.csv', '--fmt', 'xml'])
    assert r.exit_code != 0   # Choice rejects 'xml'

def test_head_invalid_n():
    r = runner.invoke(app, ['head', 'sample.csv', '--rows', '0'])
    assert r.exit_code != 0   # min=1 rejects 0

tests = [test_version, test_info, test_head_default, test_head_n2,
         test_stats_json, test_stats_invalid_fmt, test_head_invalid_n]
passed = 0
for t in tests:
    try: t(); print(f'  ✓ {t.__name__}'); passed += 1
    except AssertionError as e: print(f'  ✗ {t.__name__}: {e}')
print(f'\n{passed}/{len(tests)} passed')


**What just happened?**
- `test_stats_json` deserialises the output and asserts on the data structure — not just string presence
- `test_stats_invalid_fmt` and `test_head_invalid_n` verify **Typer's built-in validation is actually rejecting bad input** — critical regression guards
- 7 tests cover version, 3 commands, a feature option, and 2 validation failures — complete coverage in ~30 lines


---
## pyproject.toml (copy into your project root)

```toml
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "datakit"
version = "1.0.0"
requires-python = ">=3.10"
dependencies = ["typer[all]>=0.9.0"]

[project.scripts]
datakit = "datakit.main:app"

[tool.hatch.build.targets.wheel]
packages = ["src/datakit"]
```

```
# Install locally and run:
pip install -e .
datakit --help
datakit info data/sample.csv
datakit stats data/sample.csv --fmt json
```


---
## Step 3 · Production error handling

Good CLIs give clear, actionable errors. Three Typer patterns:

| Pattern | Use when |
|---|---|
| `typer.echo(msg, err=True)` + `Exit(1)` | Expected errors (file not found, bad state) |
| `typer.BadParameter(msg)` | Invalid parameter value with context |
| `typer.confirm('...', abort=True)` | Destructive operations — prompt before proceeding |


In [ ]:
err_app = typer.Typer()

@err_app.command()
def delete(
    name:  str  = typer.Argument(...),
    force: bool = typer.Option(False, '--force', '-f'),
):
    """Delete a resource by NAME."""
    if not force:
        typer.confirm(f"Delete '{name}'? Cannot be undone.", abort=True)
    typer.echo(f'Deleted: {name}')

@err_app.command()
def validate(age: int = typer.Argument(...)):
    """Validate AGE."""
    if age < 0 or age > 150:
        raise typer.BadParameter(f'Age {age} is not plausible', param_hint="'age'")
    typer.echo(f'Valid: {age}')

# --force skips confirmation
r1 = runner.invoke(err_app, ['delete', 'prod-db', '--force'])
print('--force:', r1.output.strip(), '| exit:', r1.exit_code)

# 'n' at prompt triggers Abort → exit 1
r2 = runner.invoke(err_app, ['delete', 'prod-db'], input='n\n')
print('aborted exit code:', r2.exit_code)

# BadParameter
r3 = runner.invoke(err_app, ['validate', '999'])
print('bad param exit:', r3.exit_code, '| msg:', r3.output.strip()[:50])


**What just happened?**
- **`abort=True`** on `confirm()` — user says 'n' → Typer raises `Abort`, exits 1 automatically
- `input='n\n'` in CliRunner — simulates user keyboard input; essential for testing interactive prompts
- `typer.BadParameter` formats the error identically to Typer's own validation errors — consistent UX


---
## Step 4 · Config file support (TOML)

Pattern: load `~/.datakit.toml` at startup, merge with CLI args (CLI always wins).
`tomllib` is in the Python 3.11+ stdlib — no extra dependency.


In [ ]:
import tomllib, sys
from pathlib import Path

SAMPLE_TOML = b'[defaults]\nrows = 10\nformat = "json"\n'

def load_config(path: Optional[Path] = None) -> dict:
    """Load TOML config; return {} if not found."""
    target = path or Path.home() / '.datakit.toml'
    if not target.exists():
        return {}
    with open(target, 'rb') as f:
        return tomllib.load(f)

# Parse from bytes (simulating the file)
cfg = tomllib.loads(SAMPLE_TOML.decode())
print('Config:', cfg)

# Merge pattern: CLI arg wins over config wins over code default
def resolved_rows(cli_rows: Optional[int], config: dict, default: int = 5) -> int:
    if cli_rows is not None:
        return cli_rows
    return config.get('defaults', {}).get('rows', default)

print('No CLI arg:  ', resolved_rows(None,  cfg))   # 10 from config
print('CLI arg=3:   ', resolved_rows(3,     cfg))   # 3  from CLI
print('No config:   ', resolved_rows(None,  {}))    # 5  hardcoded default


**What just happened?**
- `tomllib` stdlib (3.11+) — no extra dependency for TOML parsing
- The merge pattern is the 12-factor standard: **CLI arg → config file → code default**
- `load_config()` returns `{}` silently if no file exists — users without a config get sane defaults


---
## Step 5 · Auto-generate README from --help

Generate your README programmatically so it never drifts from the actual CLI.
Run at build time or in a `Makefile` target.


In [ ]:
def generate_readme(app_inst, tool_name: str) -> str:
    gen = CliRunner()
    top = gen.invoke(app_inst, ['--help']).output
    cmds = []
    for cmd in ['info', 'head', 'stats']:
        r = gen.invoke(app_inst, [cmd, '--help'])
        if r.exit_code == 0:
            cmds.append(f'### `{tool_name} {cmd}`\n\n```\n{r.output.strip()}\n```')
    return f'# {tool_name}\n\n## Install\n\n```bash\npip install {tool_name}\n```\n\n## Usage\n\n```\n{top.strip()}\n```\n\n## Commands\n\n' + '\n\n'.join(cmds) + '\n'

readme = generate_readme(app, 'datakit')
print(readme[:600])
print(f'README length: {len(readme)} chars')


**What just happened?**
- The README is **generated from the live `--help` output** — docs stay in sync with code automatically
- Loop over commands, capture `--help`, embed in markdown — this pattern works for any Typer app
- In a real project: `hatch run docs` or a pre-commit hook regenerates the README before every push


---
## Step 6 · Integration test: chained commands

Integration tests invoke multiple commands in sequence and assert on the combined state — validating the full user workflow, not just individual commands.


In [ ]:
# Integration test: info → head → stats in one session
def test_full_workflow():
    # Step 1: info gives row and column count
    r1 = runner.invoke(app, ['info', 'sample.csv'])
    assert r1.exit_code == 0
    assert '5' in r1.output   # 5 rows

    # Step 2: head with n=2 gives Alice but not Carol
    r2 = runner.invoke(app, ['head', 'sample.csv', '--rows', '2'])
    assert r2.exit_code == 0
    assert 'Alice' in r2.output
    assert 'Carol' not in r2.output

    # Step 3: stats JSON has numeric cols with expected keys
    r3 = runner.invoke(app, ['stats', 'sample.csv', '--fmt', 'json'])
    assert r3.exit_code == 0
    data = json.loads(r3.output)
    for col in ['age', 'salary']:
        assert col in data
        assert all(k in data[col] for k in ['min', 'max', 'mean'])

test_full_workflow()
print('Integration test ✓')


In [ ]:
# Packaging commands reference (run in terminal, not notebook)
PACKAGING = '''
# Project layout
datakit/
  pyproject.toml
  README.md
  src/datakit/__init__.py
  src/datakit/main.py    ← app = typer.Typer(); all commands here
  tests/test_commands.py

# Install in dev mode
pip install -e .

# Run CLI
datakit --help
datakit info data/sample.csv

# Run tests
pytest tests/ -v

# Build & publish
pip install hatch
hatch build
hatch publish        # requires PyPI account
'''
print(PACKAGING)


**What just happened?**
- `test_full_workflow()` chains three `runner.invoke()` calls — each builds on the previous, mirroring a real user session
- `assert all(k in data[col] for k in [...])` — structural assertion on JSON output is more robust than string matching
- The packaging reference covers the full lifecycle: dev install → test → build → publish


In [ ]:
# Demonstrate error handling in datakit context
def test_missing_file_graceful():
    """datakit should exit 1 cleanly on missing file, not crash."""
    # Override _read to raise FileNotFoundError for non-sample paths
    r = runner.invoke(app, ['info', '/nonexistent/data.csv'])
    # Should exit non-zero (file not found path in _read)
    # In production: exit_code==1; in our sample stub: may be 0 — document both
    print(f'Missing file exit code: {r.exit_code}')
    print(f'Output: {r.output.strip()[:80]}')

def test_stats_structure():
    r = runner.invoke(app, ['stats', 'sample.csv', '--fmt', 'json'])
    data = json.loads(r.output)
    # Verify statistical correctness
    assert data['age']['min'] == 25.0
    assert data['age']['max'] == 42.0
    assert data['salary']['mean'] == round((95000+72000+110000+68000+130000)/5, 2)
    print('Stats values correct ✓')

test_missing_file_graceful()
test_stats_structure()


In [ ]:
# Challenge: Add a `sample` command to datakit:
#   datakit sample FILE --rows 3 --seed 42
# Requirements:
#   - Returns N random rows (use random.sample)
#   - --seed INT option for reproducibility (default: None)
#   - Displays as a Rich table
#   - Write 2 tests: one checks reproducibility (same seed → same first row)

import random
# Your solution here


---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| `@app.callback()` | Global options (`--version`) that apply before every command |
| `[project.scripts]` in pyproject.toml | Creates the shell entry point after `pip install` |
| Test for validation failures | `assert exit_code != 0` for invalid inputs — catches regressions |
| `json.loads(result.output)` | Assert on data structure, not just string presence |
| `pip install -e .` | Editable install — code changes reflected immediately |

> **Tip:** A well-documented CLI tool on PyPI demonstrates more engineering depth than any course certificate. Invest in the README and `--help` text.

---
## 🎉 Course complete!
You've built a production-grade CLI tool with typed arguments, validation, testing, and packaging.

**Next steps:** Textual for TUI apps · Click for advanced plugin architectures · uv for fast CLI tool distribution.

Mark Day 3 complete in your [tracker](../index.html).
